# LinearSwap quickstart

Swap the linear-attention kernel of a pretrained hybrid LLM (Qwen3.5-0.8B here) in place, verify that the swap is exact, generate text, and save the result as a Hugging Face checkpoint.

Requirements: one CUDA GPU, `pip install -e .` from the repository root, and the backbone in `models/Qwen3.5-0.8B` (`huggingface-cli download Qwen/Qwen3.5-0.8B --local-dir models/Qwen3.5-0.8B`).

In [1]:
import os, pathlib
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")          # pick the GPU before torch is imported
root = pathlib.Path.cwd()
if root.name == "examples":                                   # run from the repository root
    root = root.parent
os.chdir(root); print("working directory:", root)

working directory: /mnt/yuang/gdn2-in-place


In [2]:
import torch, linswap
from linswap import build_model, list_kernels, get_kernel
from transformers import AutoTokenizer

print(list_kernels())
for k in list_kernels():
    s = get_kernel(k); print(f"{k:13s} exact_init={s.exact_init!s:5} {s.description}")

/mnt/yuang/gdn2-in-place/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Current Triton version 3.2.0 is below the recommended 3.3.0 version. Errors may occur and these issues will not be fixed. Please consider upgrading Triton.


['deltanet', 'gdn', 'gdn2', 'gla', 'kda', 'kda_fullgate', 'mamba2', 'rwkv7']
deltanet      exact_init=False DeltaNet (FLA DeltaNet, no decay); GDN weights copied, decay branch dropped — NOT function preserving.
gdn           exact_init=True  Original Gated DeltaNet (FLA GatedDeltaNet); exact weight copy, control baseline.
gdn2          exact_init=True  Gated DeltaNet-2 (FLA GatedDeltaNet2); scalar beta/decay tiled into channel-wise b/w/f gates.
gla           exact_init=False Gated Linear Attention (FLA GatedLinearAttention); shared weights copied, decay MLP at FLA init — NOT function preserving.
kda           exact_init=True  Kimi Delta Attention (FLA KimiDeltaAttention); scalar decay tiled into the low-rank per-channel gate.
kda_fullgate  exact_init=True  KDA with a dense (full-rank) f_proj decay projection instead of the low-rank MLP.
mamba2        exact_init=False Mamba-2 SSD recurrence (FLA simple-GLA kernel); GDN decay/projections copied, delta-rule erase and beta dropped — NOT fu

## 1. Build a swapped model

`build_model(kernel)` reads the backbone's `config.json`, builds the hybrid with every linear-attention layer replaced by `kernel`, and initialises the new layers from the pretrained Gated-DeltaNet weights. For *exact* kernels the model computes the same function as the original at step 0.

In [3]:
tok = AutoTokenizer.from_pretrained("models/Qwen3.5-0.8B")
model = build_model("kda", base_model_dir="models/Qwen3.5-0.8B", device="cuda").eval()
print(f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M params, "
      f"{sum(p.numel() for _, p in model.new_parameters())/1e6:.1f}M new (kernel gate) params")

759M params, 7.4M new (kernel gate) params


## 2. Check exactness against the original model

The `gdn` kernel is an exact weight copy of the original layer on FLA kernels, so it is the noise floor: an exact swap should agree with it to bf16 rounding.

In [4]:
ref = build_model("gdn", device="cuda").eval()
ids = tok("The quick brown fox jumps over the lazy dog. In 1815 the Congress of Vienna", return_tensors="pt").input_ids.cuda()
with torch.no_grad():
    a, b = model(ids).float(), ref(ids).float()
print("top-1 agreement:", (a.argmax(-1) == b.argmax(-1)).float().mean().item(), "| max |Δlogit|:", (a - b).abs().max().item())

top-1 agreement: 1.0 | max |Δlogit|: 0.34375


## 3. Generate

Greedy decoding with the KV cache for the attention layers and the recurrent state for the linear layers.

In [5]:
msgs = [{"role": "user", "content": "In one sentence, what is a linear-attention model?"}]
prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].cuda()
out = model.generate(prompt, max_new_tokens=60, eos_token_id=tok.eos_token_id)
print(tok.decode(out[0, prompt.shape[1]:], skip_special_tokens=True))

A linear-attention model is a type of transformer architecture where the input is processed by a linear layer followed by a self-attention mechanism, allowing the model to learn global dependencies while maintaining a linear computational structure.


## 4. Save as a Hugging Face checkpoint and reload with `transformers`

`linswap` registers the `linswap` architecture with the Auto classes, so the folder loads like any HF model (and can be pushed to the Hub with `push_to_hub`).

In [ ]:
from linswap.hf import LinearSwapForCausalLM
from transformers import AutoModelForCausalLM

hf = LinearSwapForCausalLM.from_swap("kda", device="cuda")
hf.save_pretrained("hf/Qwen3.5-0.8B-KDA"); tok.save_pretrained("hf/Qwen3.5-0.8B-KDA")

reloaded = AutoModelForCausalLM.from_pretrained("hf/Qwen3.5-0.8B-KDA", dtype=torch.bfloat16).cuda()
print(tok.decode(reloaded.generate(prompt, max_new_tokens=30, do_sample=False)[0, prompt.shape[1]:], skip_special_tokens=True))

## 5. Next steps

* `linswap verify --kernel kda --baseline gdn` — the full function-preservation report (layer, logits, layer-wise, cache, generation).
* `linswap posttrain --kernel kda` — gate-only and full SFT with the standard recipe.
* `linswap distill --kernel mamba2` — for kernels whose init is not exact.
* `linswap evaluate --models kda outputs/kda/sft_full/checkpoint-50 --tasks niah_multikey_2,vt,qa_1` — RULER.
* Add your own kernel: `src/linswap/kernels/gla.py` (a stock FLA layer) or `kernels/mamba2.py` (an FLA op) are the templates.